# Qwen review of explicit relations

Select a **T4 GPU runtime** and run the cells in order. When prompted, upload one review plan JSON. The plan contains the selected text excerpts and their character offsets. Qwen reviews the excerpts and the script checks the quotations against the supplied text.

Save the downloaded review in the local repository's outputs folder. Each run includes its scope, UTC timestamp and plan fingerprint in the filename. Use that file to merge the review with the graph and score it locally.

## Set up Colab

The cell updates the project checkout and installs the Qwen dependencies listed in the project requirements.

In [ ]:
from datetime import datetime, timezone
import hashlib
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys

from google.colab import files

if shutil.which("nvidia-smi") is None:
    raise RuntimeError("Select a T4 GPU runtime in Colab before running this notebook.")
subprocess.run(["nvidia-smi", "-L"], check=True)

workspace = Path("/content")
repo = workspace / "forensic-llm-graph"
if repo.exists():
    subprocess.run(["git", "pull", "--ff-only"], cwd=repo, check=True)
else:
    subprocess.run(
        ["git", "clone", "https://github.com/AntoFratta/forensic-llm-graph.git", str(repo)],
        check=True,
    )
subprocess.run(["git", "log", "-1", "--oneline"], cwd=repo, check=True)

model_packages = ("torch", "transformers", "accelerate", "bitsandbytes", "sentencepiece")
requirements = (repo / "requirements.txt").read_text(encoding="utf-8").splitlines()
dependencies = [line for line in requirements if line.startswith(model_packages)]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *dependencies], check=True)
os.chdir(workspace)

## Upload the review plan

In [ ]:
uploaded = files.upload(target_dir=str(workspace))
if len(uploaded) != 1:
    raise ValueError("Upload exactly one review plan JSON.")
plan_path = Path(next(iter(uploaded)))
if plan_path.suffix.lower() != ".json":
    raise ValueError("The review plan must be a JSON file.")

with plan_path.open(encoding="utf-8-sig") as handle:
    plan = json.load(handle)
if plan.get("run", {}).get("stage") != "relation_review_plan":
    raise ValueError("The uploaded JSON must be a relation review plan.")
print(f"Plan: {plan_path.name} ({len(plan['passages'])} passages)")

## Run Qwen

The first run in a new runtime also downloads the model weights. The script reports progress for each excerpt.

In [ ]:
plan_sha256 = hashlib.sha256(plan_path.read_bytes()).hexdigest()
scope = "corpus" if plan["run"].get("scope") == "corpus" else "diagnostic"
timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
output_path = workspace / f"relation_reviews_{scope}_{timestamp}_{plan_sha256[:8]}.json"
subprocess.run(
    [sys.executable, "-u", "scripts/review_relation_graph.py", "run",
     "--plan", str(plan_path), "--output", str(output_path)],
    cwd=repo, check=True,
)

with output_path.open(encoding="utf-8") as handle:
    result = json.load(handle)
run = result["run"]
if (run.get("stage") != "relation_review"
        or run.get("plan_sha256") != plan_sha256):
    raise ValueError(f"The output {output_path.name} does not match the uploaded plan.")
print(f"Saved review: {output_path.name}")
print(f"Review statuses: {run['statuses']}")

## Download the review output

In [ ]:
files.download(str(output_path))